# Natural Language to SQL with Ollama

This notebook demonstrates how to use an LLM via **Ollama** to translate natural language questions into valid SQL queries. The pipeline uses:

- **Pydantic** for structured output schema definition
- **Ollama's grammar-constrained decoding** to guarantee valid JSON output
- **sqlglot** for syntax validation
- A **self-correction retry loop** that feeds errors back to the model

## Prerequisites

1. Install and run [Ollama](https://ollama.com/) or use an external Ollama endpoint.
2. Pull a suitable model: `ollama pull qwen2.5-coder:1.5b`
3. (Local, not on ARC) Install Python dependencies: `pip install ollama pydantic sqlglot`

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install ollama pydantic sqlglot

## Step 0: Connect to Ollama Endpoint
If you are using an Ollama endpoint, either Ollama Cloud or a dedicated server, you need to set the endpoint's base URL and your APU key.
1. Save your API key in a file outside of your project directory, that you can load into the notebook. **Never include API keys directly into your notebooks or program code!**
2. Define the variables `OLLAMA_BASE_URL` and `OLLAMA_API_KEY` to your configuration.

The following code tests your Ollama connection by listing your installed models.

In [ ]:
import os
OLLAMA_BASE_URL="https://gpu-01.insight.gsu.edu:11443"
OLLAMA_API_KEY=open(os.path.expanduser("~/.secrets/insight_ollama_msa8700_student.txt"), 'r').read().strip()
print(f"Length of API Key: {len(OLLAMA_API_KEY)}")

# from ollama import chat
from ollama import Client

client = Client(
    host=OLLAMA_BASE_URL,
    headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'}
)

import pandas as pd
import json
response = client.list()
models = pd.DataFrame.from_records([m.model_dump() for m in response.models])
for col in ['parent_model', 'format', 'family', 'families', 'parameter_size', 'quantization_level']:
    models[col] = models.apply(lambda m: m.details[col], axis=1)
display(models[['model', 'size', 'family', 'parameter_size', 'quantization_level']])

## Step 1: Define the Structured Output Schema

We define a Pydantic model with two fields:
- `sql` — the generated SQL query
- `explanation` — a natural language description of what the query does

Separating these two fields is a deliberate design choice. The `explanation` field acts as a
chain-of-thought forcing function — the model reasons about the query intent before committing
to SQL. It also keeps the `sql` field clean of commentary, so it can be executed directly.

In [ ]:
from pydantic import BaseModel


class SQLQuery(BaseModel):
    sql: str
    explanation: str

## Step 2: Define the Database Schema (DDL)

The schema is provided in `CREATE TABLE` DDL format because this mirrors what models saw during
training and encodes relationships unambiguously. Key practices:

- **Inline comments** on columns with constrained values prevent the model from hallucinating enum values
- **Foreign key relationships** are essential for correct JOIN generation
- For large databases, only include tables relevant to the query

In [ ]:
SCHEMA_DDL = """
CREATE TABLE customers (
    customer_id SERIAL PRIMARY KEY,
    name        VARCHAR(100),
    email       VARCHAR(100),
    region      VARCHAR(50),   -- values: 'North', 'South', 'East', 'West'
    created_at  TIMESTAMP
);

CREATE TABLE orders (
    order_id    SERIAL PRIMARY KEY,
    customer_id INT REFERENCES customers(customer_id),
    amount      NUMERIC(10,2),
    status      VARCHAR(20),   -- values: 'pending', 'shipped', 'delivered'
    order_date  DATE
);

CREATE TABLE products (
    product_id  SERIAL PRIMARY KEY,
    name        VARCHAR(100),
    category    VARCHAR(50),   -- values: 'Electronics', 'Clothing', 'Books', 'Food'
    price       NUMERIC(10,2)
);

CREATE TABLE order_items (
    item_id     SERIAL PRIMARY KEY,
    order_id    INT REFERENCES orders(order_id),
    product_id  INT REFERENCES products(product_id),
    quantity    INT
);
"""

## Step 3: Construct the Prompt

A well-structured prompt has four sections:
1. **System instructions** — role, dialect, behavioral constraints
2. **Schema context** — the DDL from above
3. **Few-shot examples** — anchor the output pattern for dialect-specific syntax
4. **User question** — placed last with a SQL continuation primer (`-- SQL:`)

In [ ]:
SYSTEM_PROMPT = """You are an expert PostgreSQL query generator.
Only generate SELECT statements using the schema provided.
Never reference tables or columns not in the schema.
Use table aliases in all multi-table queries.
Do not include markdown formatting or explanation in the sql field."""

FEW_SHOT_EXAMPLES = """
-- Question: How many orders were placed last month?
-- SQL: SELECT COUNT(*) FROM orders WHERE order_date >= date_trunc('month', now() - interval '1 month') AND order_date < date_trunc('month', now());

-- Question: Top 5 customers by total spend
-- SQL: SELECT c.name, SUM(o.amount) AS total_spend FROM customers c JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id, c.name ORDER BY total_spend DESC LIMIT 5;
"""


def build_messages(user_question: str) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"""
Schema:
{SCHEMA_DDL}

Examples:
{FEW_SHOT_EXAMPLES}

-- Question: {user_question}
-- SQL:
""",
        },
    ]

## Step 4: SQL Syntax Validation with sqlglot

Ollama's structured output guarantees valid JSON with a `sql` string field — but **not** valid SQL.
A response like `"sql": "SELEKT * FORM users"` satisfies the JSON schema perfectly.

`sqlglot` parses SQL and returns dialect-aware syntax errors, which we can feed back to the model.

In [ ]:
import sqlglot


def validate_sql(sql: str, dialect: str = "postgres") -> tuple[bool, str]:
    """Validate SQL syntax using sqlglot."""
    try:
        sqlglot.parse_one(sql, dialect=dialect)
        return True, ""
    except sqlglot.errors.ParseError as e:
        return False, str(e)

In [ ]:
# Quick test: valid SQL
print(validate_sql("SELECT name FROM customers WHERE region = 'North'"))

# Quick test: invalid SQL
print(validate_sql("SELEKT name FORM customers"))

## Step 5: Generation with Self-Correction Loop

The pipeline combines structured output, syntax validation, and error feedback into a retry loop.
When the generated SQL fails validation, the exact error message is appended to the conversation
history so the model can correct its mistake on the next attempt.

Setting `temperature: 0` ensures deterministic generation on the first attempt.

In [ ]:
# MODEL = "sqlcoder"
MODEL = "qwen2.5:1.5b"
# MODEL = "qwen3.5:latest"


def generate_sql(user_question: str, max_retries: int = 3) -> SQLQuery:
    """Generate SQL from a natural language question with validation and retry."""
    messages = build_messages(user_question)

    for attempt in range(1, max_retries + 1):
        response = client.chat(
            model=MODEL,
            messages=messages,
            format=SQLQuery.model_json_schema(),
            options={"temperature": 0},
        )

        result = SQLQuery.model_validate_json(response.message.content)

        # Validate syntax
        valid, error = validate_sql(result.sql)
        status = "Valid" if valid else f"Error: {error}"
        print(f"  [Attempt {attempt}] {status}")

        if valid:
            return result

        # Feed the error back for self-correction
        messages.append({"role": "assistant", "content": response.message.content})
        messages.append(
            {
                "role": "user",
                "content": f"The SQL has a syntax error: {error}. Please fix it.",
            }
        )

    raise RuntimeError(f"Failed to generate valid SQL after {max_retries} attempts")

## Step 6: Run Examples

Let's test the pipeline with several natural language questions of increasing complexity.

In [ ]:
questions = [
    "How many customers are in the East region?",
    "What are the top 3 most expensive products?",
    "Which customers have placed orders over $500 that have been delivered?",
    "What is the total revenue per product category?",
    "Find customers who have never placed an order.",
]

In [ ]:
for q in questions:
    print(f"\n{'=' * 70}")
    print(f"Question: {q}")
    print("-" * 70)

    result = generate_sql(q)

    print(f"\nExplanation: {result.explanation}")
    print(f"\nGenerated SQL:\n{result.sql}")

## Step 7: Inspect the Structured Output

Let's look at one example in detail to see the full JSON structure returned by the model.

In [ ]:
import json

result = generate_sql("What is the average order amount per region?")
print(json.dumps(result.model_dump(), indent=2))

## Step 8 (Optional): Execute Against a Live Database

If you have a PostgreSQL instance running, you can execute the generated SQL directly.
Always connect with a **read-only role** to prevent accidental data modification.

```sql
-- Set up a read-only role in PostgreSQL:
CREATE ROLE query_agent LOGIN PASSWORD 'secret';
GRANT CONNECT ON DATABASE mydb TO query_agent;
GRANT USAGE ON SCHEMA public TO query_agent;
GRANT SELECT ON ALL TABLES IN SCHEMA public TO query_agent;
```

In [ ]:
# Uncomment and configure to run against a live database

# import psycopg2
#
# conn = psycopg2.connect(
#     host="localhost",
#     dbname="mydb",
#     user="query_agent",
#     password="secret"
# )
#
# result = generate_sql("How many orders are pending?")
# cur = conn.cursor()
# cur.execute(result.sql)
# rows = cur.fetchall()
# for row in rows:
#     print(row)
#
# cur.close()
# conn.close()

## Key Takeaways

1. **Structured output via Pydantic + Ollama's `format` parameter** guarantees the model returns parseable JSON — no regex or string stripping needed.
2. **Schema injection in DDL format** with inline comments on constrained values is the single largest lever for SQL generation quality.
3. **Few-shot examples** anchor dialect-specific syntax (e.g., `date_trunc` for PostgreSQL).
4. **sqlglot validation** catches syntax errors that structured output cannot — the JSON schema enforces structure, not SQL correctness.
5. **The self-correction loop** feeds exact error messages back to the model, enabling it to fix its own mistakes across retries.